# Day 23: Multi-Cloud Capacity Management
> *Inference Engineering* — Chapter 7.3 | Philip Kiely, Baseten Books 2026

**Layer:** Infrastructure | **Prerequisite:** Day 22 (Routing)


## Concept Overview

Multi-cloud capacity management uses multiple cloud providers and on-premises GPU clusters simultaneously, routing traffic to minimize cost and latency while maintaining availability. The key challenges are: heterogeneous GPU availability (spot vs reserved vs on-demand), varying latency by region, and cost optimization across providers. A capacity manager maintains a model of available capacity across providers and routes requests based on a combination of cost, latency, and SLO requirements. On-premises GPUs (like home lab DGX Sparks) provide a cost-efficient base load, with cloud GPUs handling bursts.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Optional

print('Multi-cloud capacity management simulation')


## 1. Capacity Pool Model

Model available GPU capacity across multiple providers with different cost and latency profiles.


In [ ]:
@dataclass
class CapacityPool:
    name: str
    provider: str
    region: str
    gpu_type: str
    num_gpus: int
    cost_per_gpu_hr: float
    latency_ms: float  # network latency to this pool
    availability: float  # 0-1, fraction of time available (spot has lower availability)
    is_spot: bool

pools = [
    CapacityPool('spark-cluster', 'on-prem', 'local', 'GB10', 2, 0.0, 0.5, 0.99, False),
    CapacityPool('aws-us-east-1', 'AWS', 'us-east-1', 'H100', 8, 10.0, 25.0, 0.99, False),
    CapacityPool('aws-spot', 'AWS', 'us-east-1', 'H100', 16, 4.0, 25.0, 0.70, True),
    CapacityPool('gcp-us-central', 'GCP', 'us-central1', 'H100', 4, 9.5, 30.0, 0.99, False),
    CapacityPool('azure-eastus', 'Azure', 'eastus', 'H100', 4, 9.8, 22.0, 0.99, False),
    CapacityPool('lambda-us', 'Lambda', 'us-west-2', 'H100', 8, 7.5, 45.0, 0.95, False),
]

print(f'{"Pool":<20} {"Provider":<10} {"GPUs":>6} {"$/GPU/hr":>10} {"Latency":>10} {"Avail":>8} {"Spot":>6}')
print('-' * 73)
for p in pools:
    print(f'{p.name:<20} {p.provider:<10} {p.num_gpus:>6} {p.cost_per_gpu_hr:>10.1f} {p.latency_ms:>9.1f}ms {p.availability:>8.0%} {str(p.is_spot):>6}')

total_gpus = sum(p.num_gpus for p in pools)
hourly_cost = sum(p.num_gpus * p.cost_per_gpu_hr for p in pools)
print(f'\nTotal GPU capacity: {total_gpus} GPUs')
print(f'Hourly cost if all active: ${hourly_cost:.0f}/hr')


## 2. Multi-Cloud Routing Policy

Route requests to the cheapest available pool that meets the latency SLO.


In [ ]:
def route_request(pools, request_type, latency_slo_ms):
    """
    Route to cheapest available pool meeting the latency SLO.
    request_type: 'interactive' | 'batch'
    """
    available = [p for p in pools if p.availability > 0.9 or not p.is_spot]
    within_slo = [p for p in available if p.latency_ms <= latency_slo_ms]

    if not within_slo:
        within_slo = available  # relax SLO if nothing available

    if request_type == 'batch':
        # For batch: minimize cost, tolerate higher latency
        return min(within_slo, key=lambda p: p.cost_per_gpu_hr)
    else:
        # For interactive: minimize latency among affordable pools
        affordable = [p for p in within_slo if p.cost_per_gpu_hr < 10.0]
        if affordable:
            return min(affordable, key=lambda p: p.latency_ms)
        return min(within_slo, key=lambda p: p.latency_ms)

# Simulate routing decisions
print('Routing decisions:')
for req_type, slo in [('interactive', 50), ('interactive', 30), ('batch', 1000), ('batch', 200)]:
    pool = route_request(pools, req_type, slo)
    print(f'  {req_type:<15} SLO={slo}ms → {pool.name:<20} ({pool.provider}, ${pool.cost_per_gpu_hr}/hr, {pool.latency_ms}ms)')


## 3. Cost Optimization — On-Prem as Base Load

On-premises hardware handles predictable base load; cloud handles bursts. This hybrid strategy minimizes total cost.


In [ ]:
def cost_analysis(traffic_profile, on_prem_gpus=2, on_prem_cost_hr=0):
    """
    Compare pure-cloud vs hybrid cost for a traffic profile.
    traffic_profile: list of (hour, requests_per_second)

    Returns (cloud_only_cost, hybrid_cost, savings)
    """
    # Assume 1 H100 handles 10 req/s for typical LLM
    rps_per_gpu = 10.0
    cloud_cost_per_gpu_hr = 10.0  # $/hr for H100

    cloud_only_cost = 0
    hybrid_cost = 0

    for hour, rps in traffic_profile:
        gpus_needed = int(np.ceil(rps / rps_per_gpu))
        # Cloud only
        cloud_only_cost += gpus_needed * cloud_cost_per_gpu_hr
        # Hybrid: on-prem handles up to on_prem_gpus worth
        cloud_gpus = max(0, gpus_needed - on_prem_gpus)
        hybrid_cost += cloud_gpus * cloud_cost_per_gpu_hr + on_prem_cost_hr

    savings = cloud_only_cost - hybrid_cost
    return cloud_only_cost, hybrid_cost, savings

# 24-hour traffic profile (requests per second)
hours = list(range(24))
# Typical daily pattern: low at night, peaks in morning/evening
rps_profile = [
    (0, 5), (1, 3), (2, 2), (3, 2), (4, 3), (5, 5),
    (6, 10), (7, 18), (8, 25), (9, 28), (10, 30), (11, 32),
    (12, 35), (13, 33), (14, 30), (15, 28), (16, 30), (17, 32),
    (18, 38), (19, 35), (20, 28), (21, 22), (22, 15), (23, 8),
]

cloud_cost, hybrid_cost, savings = cost_analysis(rps_profile, on_prem_gpus=2)
print(f'Daily Cost Comparison (2 on-prem GPUs as base load):')
print(f'  Cloud-only:  ${cloud_cost:.0f}/day')
print(f'  Hybrid:      ${hybrid_cost:.0f}/day')
print(f'  Savings:     ${savings:.0f}/day ({savings/cloud_cost*100:.0f}%)')

# Plot traffic and cost
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
rps_vals = [r[1] for r in rps_profile]
gpus_needed = [int(np.ceil(r/10)) for r in rps_vals]
on_prem_contribution = [min(2, g) for g in gpus_needed]
cloud_contribution = [max(0, g-2) for g in gpus_needed]

ax1.stackplot(hours, on_prem_contribution, cloud_contribution,
              labels=['On-prem (DGX Spark)', 'Cloud burst'], alpha=0.7)
ax1.set_ylabel('GPUs Active'); ax1.legend()
ax1.set_title('Hybrid Multi-Cloud Capacity: On-Prem Base + Cloud Burst')

cost_cloud = [g * 10 for g in gpus_needed]
cost_hybrid = [c * 10 for c in cloud_contribution]
ax2.plot(hours, cost_cloud, label='Cloud-only cost', color='red')
ax2.plot(hours, cost_hybrid, label='Hybrid cost', color='green')
ax2.fill_between(hours, cost_hybrid, cost_cloud, alpha=0.3, color='green', label='Savings')
ax2.set_xlabel('Hour of Day'); ax2.set_ylabel('$/hr'); ax2.legend()
plt.tight_layout()
plt.savefig('multi_cloud_cost.png', dpi=100, bbox_inches='tight')
plt.show()


## Experiments: Try These

1. **Spot instance strategy**: Add spot instance interruption simulation. When a spot pool goes down, measure the failover time and traffic loss.
2. **Geo-latency routing**: Add user geolocation to requests and route to the nearest pool meeting cost SLO. How much latency does geo-routing save?
3. **Capacity reservation optimization**: Given a budget of $X/day, find the optimal mix of reserved vs on-demand vs spot capacity to minimize expected cost.


## Key Takeaways

- Multi-cloud capacity management routes traffic across providers by combining cost, latency, and availability constraints.
- On-premises GPUs as base load with cloud burst is the most cost-efficient pattern: predictable workload at zero marginal cost, burst handled by cloud.
- Spot instances can cut GPU costs by 60-70% but require fallback to on-demand when interrupted — build bidding and fallback into the capacity manager.
- Route interactive traffic by latency, batch traffic by cost — the same model with two queues and two routing policies.

## References
- *Inference Engineering* Ch 7.3 — Philip Kiely, Baseten Books 2026
- AWS EC2 Spot Instances best practices
- Multi-cloud LLM serving case studies
